# 05 — Statistical Analysis & Hypothesis Testing

**Methods Covered:**
1. Descriptive statistics (mean, std, median, IQR, skewness, kurtosis).
2. Missing-value and outlier analysis (Tukey IQR method + Z-score audit).
3. Cross-state correlation analysis between BEE public charging station count and cumulative EV sales (Pearson $r$ and Spearman $\rho$).
4. Hypothesis Testing:
   - **H1 (Mann-Whitney U):** Do verified highway corridor hubs have significantly higher rated `max_power_kw` than unverified OSM-listed points? (Non-parametric test chosen because `max_power_kw` is right-skewed with small subgroup sample sizes.)
   - **H2 (Chi-Square Test of Independence):** Is `charging_type` (`DC_FAST`/`MIXED` vs `AC`/`SWAP_OR_2W`) independent of region (`Bengaluru Urban` vs `Rest of Karnataka / Highways`)?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

REPO = Path("..").resolve()
PROC = REPO / "data" / "processed"
RAW = REPO / "data" / "raw" / "india_ev_market"

stations = pd.read_csv(PROC / "karnataka_stations_clean.csv")
india_sales = pd.read_csv(PROC / "india_ev_sales_clean.csv")
bee_stations = pd.read_csv(RAW / "dim_public_charging_stations_by_state.csv")

# 1. Descriptive Statistics & Distribution Shape
pw = stations["max_power_kw"].dropna()
q1, q3 = pw.quantile(0.25), pw.quantile(0.75)
iqr = q3 - q1
print("=== Rated Power (kW) Descriptive Stats ===")
print(f"n={len(pw)} | mean={pw.mean():.2f} | std={pw.std():.2f} | median={pw.median():.2f} | IQR={iqr:.2f}")
print(f"skewness={stats.skew(pw):.2f} | kurtosis={stats.kurtosis(pw):.2f}")
outliers_pw = pw[(pw < q1 - 1.5 * iqr) | (pw > q3 + 1.5 * iqr)]
print(f"Tukey IQR outliers in max_power_kw: {len(outliers_pw)} -> {outliers_pw.tolist()}")

ka_2w = india_sales[(india_sales["state"] == "Karnataka") & (india_sales["vehicle_category"] == "2-Wheelers")]["electric_vehicles_sold"]
z_scores = np.abs(stats.zscore(ka_2w))
print(f"\nKarnataka monthly 2W EV sales mean={ka_2w.mean():.1f}, median={ka_2w.median():.1f}, skew={stats.skew(ka_2w):.2f}, |Z|>2.5 count={(z_scores > 2.5).sum()}")

=== Rated Power (kW) Descriptive Stats ===
n=17 | mean=47.67 | std=31.27 | median=60.00 | IQR=35.00
skewness=0.39 | kurtosis=-0.07
Tukey IQR outliers in max_power_kw: 1 -> [120.0]

Karnataka monthly 2W EV sales mean=8120.2, median=7740.0, skew=0.75, |Z|>2.5 count=1


In [2]:
# 2. Cross-State Correlation: Public Charging Stations vs Total EV Sales
state_totals = india_sales.groupby("state", as_index=False)["electric_vehicles_sold"].sum()
merged_state = pd.merge(bee_stations, state_totals, on="state", how="inner")

pearson_r, pearson_p = stats.pearsonr(merged_state["public_charging_stations"], merged_state["electric_vehicles_sold"])
spearman_rho, spearman_p = stats.spearmanr(merged_state["public_charging_stations"], merged_state["electric_vehicles_sold"])
print(f"Cross-state correlation (n={len(merged_state)} states/UTs):")
print(f"  Pearson r  = {pearson_r:.4f} (p = {pearson_p:.4e})")
print(f"  Spearman rho = {spearman_rho:.4f} (p = {spearman_p:.4e})")
print("Conclusion: Strong positive association between state public charging infrastructure and cumulative EV sales (correlation does not imply solo causation; income & subsidies also drive both).")

Cross-state correlation (n=34 states/UTs):
  Pearson r  = 0.8408 (p = 4.8731e-10)
  Spearman rho = 0.9507 (p = 7.8010e-18)
Conclusion: Strong positive association between state public charging infrastructure and cumulative EV sales (correlation does not imply solo causation; income & subsidies also drive both).


In [3]:
# 3. Hypothesis Test H1: Power rating of Verified Corridor Stations vs OSM-Listed Stations
ver_pw = stations[stations["is_verified_tier"]]["max_power_kw"].dropna()
unv_pw = stations[~stations["is_verified_tier"]]["max_power_kw"].dropna()
print(f"Verified known power n={len(ver_pw)} (median={ver_pw.median():.1f} kW) vs Unverified OSM known power n={len(unv_pw)} (median={unv_pw.median():.1f} kW)")
if len(ver_pw) >= 3 and len(unv_pw) >= 3:
    u_stat, u_p = stats.mannwhitneyu(ver_pw, unv_pw, alternative="greater")
    print(f"Mann-Whitney U = {u_stat:.1f}, one-sided p = {u_p:.4f}")

# 4. Hypothesis Test H2: Chi-Square test of Charging Category vs Region
stations["is_car_dc"] = stations["charging_type"].isin(["DC_FAST", "MIXED"])
contingency = pd.crosstab(stations["region_group"], stations["is_car_dc"])
display(contingency)
chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square = {chi2:.3f}, dof = {dof}, p = {chi_p:.4e}")
print("Conclusion: Highway/non-Bengaluru corridor records in ChargeWise have a statistically significant higher concentration of car DC fast chargers than Bengaluru OSM points (which include many 2-wheeler/swap nodes).")

Verified known power n=15 (median=60.0 kW) vs Unverified OSM known power n=2 (median=3.0 kW)


is_car_dc,False,True
region_group,,
Bengaluru Urban,16,39
Rest of Karnataka / Highways,2,29


Chi-square = 4.848, dof = 1, p = 2.7675e-02
Conclusion: Highway/non-Bengaluru corridor records in ChargeWise have a statistically significant higher concentration of car DC fast chargers than Bengaluru OSM points (which include many 2-wheeler/swap nodes).
